# ML-07 — Baseline Action Score and Top-20 Review

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/T0othIess/FlyRank-AI-ML-internship/blob/main/work/notebooks/w04_baseline_score.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. My rule and its reason codes

*Write the rule in plain words first. Then the reason codes it can output.*  
**rule:** score pages by ctr_gap: the difference between expected CTR for its position and the CTR its getting. the bigger the gap the more the page needs reviewing  
**reason code:** LOW_CTR_FOR_POSITION  
**Action:** review  

**side note:** competition level turned out to be against what i expected, i thought the higher the competition the lower the CTR because there would be strong candidates, it turns out to be the opposite.

**Signal 1 (position vs CTR):** CONFIRMED — CTR increases as position improves (beyond20: 0.13%, top_11to20: 0.31%, top_10: 0.34%)  

**Signal 2 (competition_level vs CTR):** OPPOSITE — CTR slightly increases with competition, opposite of my hunch that higher competition would lower CTR

In [1]:
import os
from huggingface_hub import login
import duckdb
import pandas as pd

HF_TOKEN = os.environ.get("HF_TOKEN")
login(HF_TOKEN)
con = duckdb.connect()
con.execute(f"CREATE OR REPLACE SECRET hf (TYPE huggingface, TOKEN '{HF_TOKEN}')")
rel = "hf://datasets/FlyRank/internship-warehouse"
fact_content_daily_performance_table = con.sql(f"SELECT * FROM read_parquet('{rel}/fact_content_daily_performance/month=2026-03/*.parquet')")
dim_content_table  = con.sql(f"SELECT * from read_parquet('{rel}/dim_content.parquet')")

con.sql("""SELECT
           CASE
                WHEN gsc_avg_position <= 10 THEN 'top_10'
                WHEN gsc_avg_position <=20 THEN 'top_11to20'
                ELSE 'beyond20'
            END AS position_tier, COUNT(*) AS n, SUM(gsc_clicks) * 1.0 / SUM(NULLIF(gsc_impressions,0)) AS ctr

            from fact_content_daily_performance_table
            WHERE gsc_data_available IS TRUE AND gsc_avg_position >0
            GROUP BY position_tier
            ORDER BY ctr
        """).show()

con.sql("""SELECT d.competition_level, COUNT(*) AS n, SUM(f.gsc_clicks) * 1.0 / SUM(NULLIF(f.gsc_impressions, 0)) AS ctr
           FROM fact_content_daily_performance_table AS f JOIN dim_content_table AS d USING(content_hash_id)
           WHERE f.gsc_data_available IS TRUE AND d.is_deleted IS FALSE AND f.gsc_avg_position > 0
           GROUP BY d.competition_level
           ORDER BY ctr""").show()

c:\Users\M-H-M-D\AppData\Local\Python\pythoncore-3.14-64\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Note: Environment variable`HF_TOKEN` is set and is the current active token independently from the token you've just configured.


┌───────────────┬─────────┬───────────────────────┐
│ position_tier │    n    │          ctr          │
│    varchar    │  int64  │        double         │
├───────────────┼─────────┼───────────────────────┤
│ beyond20      │  908354 │  0.001314496204492777 │
│ top_11to20    │  519223 │ 0.0031460212728466742 │
│ top_10        │ 2020295 │  0.003396787358141528 │
└───────────────┴─────────┴───────────────────────┘

┌───────────────────┬─────────┬──────────────────────┐
│ competition_level │    n    │         ctr          │
│      varchar      │  int64  │        double        │
├───────────────────┼─────────┼──────────────────────┤
│ LOW               │ 2798111 │ 0.002900155837336515 │
│ MEDIUM            │  235994 │  0.00296847541901573 │
│ HIGH              │  289982 │  0.00321123680132192 │
│ NULL              │  122016 │ 0.003932982428846769 │
└───────────────────┴─────────┴──────────────────────┘



## 2. Build the ranked queue (writes the CSV)

*Code the score, rank everything, write work/outputs/baseline_action_score.csv.*

In [2]:
df = con.sql("""SELECT f.content_hash_id, f.gsc_clicks, f.gsc_impressions, f.gsc_avg_position, d.search_volume, d.competition_level, d.main_intent
             FROM fact_content_daily_performance_table AS f JOIN dim_content_table AS d USING(content_hash_id)
             WHERE f.gsc_data_available IS TRUE AND d.is_deleted IS FALSE AND f.gsc_avg_position > 0 AND d.search_volume IS NOT NULL""").df()

df["main_intent"] = df["main_intent"].astype("category")

competition_ranks = pd.api.types.CategoricalDtype(categories=["LOW", "MEDIUM", "HIGH"], ordered=True)
df["competition_level"] = df["competition_level"].astype(competition_ranks)
df["position_tier"] = pd.cut(df["gsc_avg_position"], bins=[0,10,20,float("inf")], labels=["page_1", "striking", "page_3_5"])
expected_ctr_per_tier = df.groupby("position_tier")["gsc_clicks"].sum() / df.groupby("position_tier")["gsc_impressions"].sum()
df["expected_ctr"] = df["position_tier"].map(expected_ctr_per_tier).astype(float)
df["ctr"] = df["gsc_clicks"] / df["gsc_impressions"].mask(df["gsc_impressions"] == 0)
df["ctr_gap"] = df["expected_ctr"] - df["ctr"]
df["reason_code"] = "LOW_CTR_FOR_POSITION"
df["action"] = "review"

#the raeson index is false is because pandas by default have their first column as literal indices (0,1,2,..) which is unneeded
df.sort_values("ctr_gap", ascending=False).to_csv("../outputs/baseline_action_score.csv", index=False)


## 3. Top-20 review

*For each of the top 20: action, reason code, confidence note, and what would make it wrong.*

In [3]:
df_best = df.sort_values("ctr_gap", ascending=False).drop_duplicates(subset="content_hash_id", keep="first")
df_best.head(10).style.format("{:.3f}", subset=["expected_ctr", "ctr", "ctr_gap"]).set_properties(**{"text-align": "center"})

,content_hash_id,gsc_clicks,gsc_impressions,gsc_avg_position,search_volume,competition_level,main_intent,position_tier,expected_ctr,ctr,ctr_gap,reason_code,action
3338198,content_bd532e08b02200bb,0,1,7.000000,0,LOW,informational,page_1,0.003,0.000,0.003,LOW_CTR_FOR_POSITION,review
0,content_2e6360ad20fd7107,0,4,3.500000,10,LOW,informational,page_1,0.003,0.000,0.003,LOW_CTR_FOR_POSITION,review
1,content_ac8663da7484669a,0,8,5.875000,10,LOW,informational,page_1,0.003,0.000,0.003,LOW_CTR_FOR_POSITION,review
2,content_39d7361b4945d504,0,12,4.333333,10,LOW,informational,page_1,0.003,0.000,0.003,LOW_CTR_FOR_POSITION,review
3,content_d49a012dcb924e31,0,5,1.400000,10,LOW,informational,page_1,0.003,0.000,0.003,LOW_CTR_FOR_POSITION,review
4,content_614baf2af4330bd7,0,21,3.476190,10,LOW,informational,page_1,0.003,0.000,0.003,LOW_CTR_FOR_POSITION,review
5,content_4a1ca0fa5c177e0c,0,1,6.000000,10,LOW,informational,page_1,0.003,0.000,0.003,LOW_CTR_FOR_POSITION,review
3338182,content_fd1993182cb72733,0,1,4.000000,10,LOW,transactional,page_1,0.003,0.000,0.003,LOW_CTR_FOR_POSITION,review
3338181,content_5493162e82b21f36,0,2,4.000000,30,LOW,transactional,page_1,0.003,0.000,0.003,LOW_CTR_FOR_POSITION,review
3338180,content_22052f29b05c3a15,0,1,4.000000,20,LOW,informational,page_1,0.003,0.000,0.003,LOW_CTR_FOR_POSITION,review


**1-bd532e08b02200bb —** review; here because pos 7 (page 1) got 0/1 clicks; wrong if that 1 impression is just noise.  
**2-2e6360ad20fd7107 —** review; here because pos 3.5 got 0/4 clicks; wrong if 4 impressions is too small a sample.  
**3-ac8663da7484669a —** review; here because pos 5.9 got 0/8 clicks; wrong if a lucky click or two on 8 impressions would flip it.  
**4-39d7361b4945d504 —** review; here because pos 4.3 got 0/12 clicks; wrong since 12 impressions still isn't a large enough sample.  
**5-d49a012dcb924e31 —** review; here because pos 1.4 got 0/5 clicks; wrong if 5 impressions is too few to trust a 0% CTR.  
**6-614baf2af4330bd7 —** review; here because pos 3.5 got 0/21 clicks, the largest sample in this top 10; wrong if it doesn't hold up over more days.  
**7-4a1ca0fa5c177e0c —** review; here because pos 6.0 got 0/1 clicks; wrong since 1 impression is basically a coin flip.  
**8-fd1993182cb72733 —** review; here because pos 4.0 got 0/1 clicks on a transactional page; wrong since sample size is still just 1.  
**9-5493162e82b21f36 —** review; here because pos 4.0 got 0/2 clicks with the highest search_volume (30) in this top 10; wrong if 2 impressions still isn't enough to confirm.  
**10-22052f29b05c3a15 —** review; here because pos 4.0 got 0/1 clicks; wrong for the same low-impression reason as most others here.  

## 4. Weak picks + leakage check

*Which picks look wrong and why? Confirm no product flags or future windows leaked in.*

**weak pics:** all the top 10 share the same weakness, which is low impressions (1-21), mostly being on the very low side (1-2), so its more of a noise rather than actual predictions,
it needs a minimum impressions filter (like atleast 1000 or something) to have useful predictions, because with those current top 10, a single click would shift it all.  

**leakage check:** there is no data outside the date, all data are on 2026-03 (as proven in w03_data_contract.ipynb), also there is no label-dervied features, gsc_clicks and gsc_impressions are only used to calculate ctr_gap, never used as a score.

In [5]:
#to prove the date claim again
con.sql("SELECT MIN(report_date) AS earliest_date, MAX(report_date) AS latest_date from fact_content_daily_performance_table").show()

┌───────────────┬─────────────┐
│ earliest_date │ latest_date │
│     date      │    date     │
├───────────────┼─────────────┤
│ 2026-03-01    │ 2026-03-31  │
└───────────────┴─────────────┘



## Self-check

Before you submit, confirm each line honestly:

- [ x ] Every section above is filled — markdown thinking AND the code that backs it
- [ x ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ x ] No client names, URLs, or private queries anywhere
- [ x ] My claims use careful words: observed, measured, directional, decision-support
- [ x ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.